# Advanced PySpark Optimization — Examples 41–50

Notebook 5 of 5 — the final notebook in the Advanced PySpark Optimization series.

## Examples

41. AQE off vs. AQE on
42. Adaptive shuffle partition coalescing
43. AQE skew join handling
44. Dynamic partition pruning
45. Runtime filtering and Bloom-filter concepts
46. End-to-end AQE optimization
47. Window-function optimization
48. Complex plan optimization
49. Production troubleshooting workflow
50. End-to-end optimization case study

## Important lab note

The CSV inputs contain only 10 rows, intentionally.

AQE, skew handling, dynamic partition pruning, runtime filtering, and adaptive coalescing are **production-scale optimizations**. On tiny local data, Spark may not visibly trigger every adaptive behavior.

This notebook therefore separates:

- configuration and mechanics we can demonstrate directly
- execution-plan inspection
- realistic production diagnostics
- phenomena that require larger data to observe reliably

Do not interpret local timings from these examples as production benchmarks.

## Source mapping

This notebook completes the optimization progression by combining advanced adaptive execution, runtime filtering, query-plan optimization, window execution, and production troubleshooting concepts.

The final examples are deliberately more workflow-oriented: rather than learning one API at a time, they combine several earlier techniques and show how an experienced Spark engineer moves from:

**symptom → evidence → execution plan → targeted optimization → validation**

In [ ]:
from pathlib import Path
import shutil
import time

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

BASE_PATH = Path.cwd()
DATA_PATH = BASE_PATH / "data"

if not DATA_PATH.exists():
    candidate = Path("/mnt/data/pyspark_examples_41_50/data")
    if candidate.exists():
        DATA_PATH = candidate

assert DATA_PATH.exists(), "Could not find data/. Update BASE_PATH."

WORK_PATH = BASE_PATH / "work"
if str(BASE_PATH).startswith("/mnt/data/pyspark_examples_41_50"):
    WORK_PATH = BASE_PATH / "work"

if WORK_PATH.exists():
    shutil.rmtree(WORK_PATH)
WORK_PATH.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Advanced-PySpark-Examples-41-50")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.warehouse.dir", str(WORK_PATH / "warehouse"))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("brand", StringType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
    StructField("region", StringType(), True),
    StructField("order_year", StringType(), True),
    StructField("order_month", StringType(), True),
])

customers_df = spark.read.option("header", True).schema(customers_schema).csv(str(DATA_PATH / "customers.csv"))
products_df = spark.read.option("header", True).schema(products_schema).csv(str(DATA_PATH / "products.csv"))
orders_df = spark.read.option("header", True).schema(orders_schema).csv(str(DATA_PATH / "orders.csv"))

print("Spark version:", spark.version)
print("Orders:", orders_df.count())
print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# Example 41 — AQE Off vs. AQE On

**Concepts:** Adaptive Query Execution (AQE), runtime statistics, adaptive planning.

AQE allows Spark to revise parts of the physical plan using information collected during execution.

This example runs the same aggregation with AQE disabled and enabled.

Because the dataset is tiny, the result may look similar. The objective is to establish the comparison workflow.

In [ ]:
original_aqe = spark.conf.get("spark.sql.adaptive.enabled")

def build_aqe_query():
    return (
        orders_df
        .repartition(8, "region")
        .groupBy("region")
        .agg(
            F.sum("amount").alias("total_amount"),
            F.count("*").alias("order_count")
        )
    )

try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    aqe_off_df = build_aqe_query()
    print("AQE OFF PLAN")
    aqe_off_df.explain("formatted")
    aqe_off_result = aqe_off_df.collect()

    spark.conf.set("spark.sql.adaptive.enabled", "true")
    aqe_on_df = build_aqe_query()
    print("\nAQE ON PLAN (before/while execution)")
    aqe_on_df.explain("formatted")
    aqe_on_result = aqe_on_df.collect()

    print("\nResults identical:", sorted(aqe_off_result) == sorted(aqe_on_result))

finally:
    spark.conf.set("spark.sql.adaptive.enabled", original_aqe)

# Example 42 — Adaptive Shuffle Partition Coalescing

**Concepts:** shuffle partition count, adaptive coalescing, avoiding too many small tasks.

A fixed `spark.sql.shuffle.partitions` value can create too many tiny shuffle partitions.

With AQE enabled, Spark can coalesce post-shuffle partitions.

On this tiny dataset, inspect configuration and plan behavior rather than expecting a dramatic reduction.

In [ ]:
original_aqe = spark.conf.get("spark.sql.adaptive.enabled")
original_coalesce = spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")

try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

    coalesce_df = (
        orders_df
        .repartition(8, "region")
        .groupBy("region")
        .agg(F.sum("amount").alias("total_amount"))
    )

    print("Configured shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
    print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
    print("Adaptive coalescing:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))

    print("\nPLAN")
    coalesce_df.explain("formatted")

    print("\nRESULT")
    coalesce_df.collect()

    print("\nLab note:")
    print("Inspect the executed plan / Spark UI on larger data to observe adaptive partition coalescing.")

finally:
    spark.conf.set("spark.sql.adaptive.enabled", original_aqe)
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", original_coalesce)

# Example 43 — AQE Skew Join Handling

**Concepts:** adaptive skew handling, skewed shuffle partitions, AQE configuration.

AQE can detect skewed shuffle partitions and split/rebalance work in suitable joins.

A 10-row dataset cannot reliably exceed skew thresholds, so this example:

- creates a repeated/hot customer key
- enables AQE skew handling
- inspects the join plan
- documents what to observe on production-sized data

In [ ]:
original_settings = {
    key: spark.conf.get(key)
    for key in [
        "spark.sql.adaptive.enabled",
        "spark.sql.adaptive.skewJoin.enabled",
    ]
}

try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

    print("Customer frequency:")
    orders_df.groupBy("customer_id").count().orderBy(F.desc("count")).show()

    skew_candidate_df = (
        orders_df
        .repartition(8, "customer_id")
        .join(customers_df, "customer_id")
    )

    print("\nAQE SKEW-JOIN CANDIDATE PLAN")
    skew_candidate_df.explain("formatted")
    skew_candidate_df.collect()

    print("\nProduction observation checklist:")
    print("- Look for one or a few unusually large shuffle partitions.")
    print("- Look for long-running straggler tasks.")
    print("- Inspect the final adaptive plan after execution.")
    print("- Confirm skew handling before adding manual salting.")

finally:
    for key, value in original_settings.items():
        spark.conf.set(key, value)

# Example 44 — Dynamic Partition Pruning

**Concepts:** dynamic partition pruning (DPP), join-driven pruning, partitioned fact data.

We write orders partitioned by `region`, then join/filter through the customer side.

DPP can use values discovered from another part of the query to avoid reading irrelevant fact partitions.

Whether DPP visibly appears in this tiny local example depends on Spark version and planning decisions.

In [ ]:
partitioned_orders_path = WORK_PATH / "orders_partitioned_by_region"

(
    orders_df
    .write
    .mode("overwrite")
    .partitionBy("region")
    .parquet(str(partitioned_orders_path))
)

fact_df = spark.read.parquet(str(partitioned_orders_path))

east_customer_ids_df = (
    customers_df
    .filter(F.col("city").isin("New York", "Boston"))
    .select("customer_id")
)

dpp_candidate_df = (
    fact_df
    .join(east_customer_ids_df, "customer_id")
    .select("order_id", "customer_id", "amount", "region")
)

print("DPP CANDIDATE PLAN")
dpp_candidate_df.explain("formatted")

print("\nRESULT")
dpp_candidate_df.show()

print("\nLook for dynamic partition pruning information in the scan/subquery sections when running larger production data.")

# Example 45 — Runtime Filtering and Bloom-Filter Concepts

**Concepts:** runtime filtering, Bloom filters, approximate membership filtering.

Spark versions and configurations differ in how runtime filtering/Bloom-filter optimizations are exposed, so this example demonstrates the **problem pattern** without assuming a particular physical operator.

We first reduce the dimension table to relevant product IDs, then join.

Conceptually, runtime filtering aims to avoid unnecessary work on rows/partitions that cannot match the join.

In [ ]:
relevant_products_df = (
    products_df
    .filter(F.col("category") == "Electronics")
    .select("product_id")
)

runtime_filter_candidate_df = (
    orders_df
    .join(relevant_products_df, "product_id")
    .select("order_id", "product_id", "amount", "region")
)

print("FILTER/JOIN PLAN")
runtime_filter_candidate_df.explain("formatted")

print("\nRESULT")
runtime_filter_candidate_df.show()

print("\nConceptual flow:")
print("1. Determine a set of relevant join keys from the smaller side.")
print("2. Use that information as early as possible to reduce unnecessary processing.")
print("3. Runtime filters and Bloom filters are implementation techniques for this general goal.")
print("4. Exact availability and operators depend on Spark version/configuration/data source.")

# Example 46 — End-to-End AQE Optimization

**Concepts:** adaptive execution, broadcast opportunities, shuffle reduction, plan inspection.

This combines multiple earlier concepts into one query:

- filter early
- join a fact-like table to small dimensions
- aggregate
- inspect the adaptive plan

We compare a baseline-style query and an optimized-style query using explicit broadcast where appropriate.

Both must return identical business results.

In [ ]:
baseline_df = (
    orders_df
    .join(customers_df, "customer_id")
    .join(products_df, "product_id")
    .filter((F.col("amount") > 300) & (F.col("category") == "Electronics"))
    .groupBy("region", "segment")
    .agg(F.sum("amount").alias("revenue"))
)

optimized_df = (
    orders_df
    .filter(F.col("amount") > 300)
    .join(F.broadcast(products_df.filter(F.col("category") == "Electronics")), "product_id")
    .join(F.broadcast(customers_df), "customer_id")
    .groupBy("region", "segment")
    .agg(F.sum("amount").alias("revenue"))
)

print("BASELINE PLAN")
baseline_df.explain("formatted")

print("\nOPTIMIZED-STYLE PLAN")
optimized_df.explain("formatted")

baseline_rows = sorted(baseline_df.collect())
optimized_rows = sorted(optimized_df.collect())

print("\nResults identical:", baseline_rows == optimized_rows)
print("Result:", optimized_rows)

# Example 47 — Window-Function Optimization

**Concepts:** window partitioning, ordering cost, sort/shuffle awareness.

Window functions often require partitioning and ordering.

We compute:

- row number
- running total
- previous order amount

all using the same window specification so Spark can potentially share compatible work.

Inspect the plan for exchanges and sorts.

In [ ]:
orders_with_date = orders_df.withColumn("order_date", F.to_date("order_date"))

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

ranking_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
)

window_df = (
    orders_with_date
    .withColumn("row_number", F.row_number().over(ranking_window))
    .withColumn("running_amount", F.sum("amount").over(customer_window))
    .withColumn("previous_amount", F.lag("amount").over(ranking_window))
)

print("WINDOW PLAN")
window_df.explain("formatted")

window_df.orderBy("customer_id", "order_date").show()

# Example 48 — Complex Plan Optimization

**Concepts:** filter early, projection pruning, avoiding repeated work, broadcast dimensions, plan simplification.

We start with a deliberately more complex analytical query and then rewrite it into a cleaner optimization-oriented version.

The optimized version:

- selects only needed columns
- filters early
- avoids carrying unnecessary columns through joins
- broadcasts small dimensions
- validates the final result

In [ ]:
complex_baseline_df = (
    orders_df
    .join(customers_df, "customer_id")
    .join(products_df, "product_id")
    .withColumn("high_value", F.col("amount") >= 800)
    .filter(F.col("high_value"))
    .filter(F.col("segment") == "Premium")
    .groupBy("city", "brand")
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("orders")
    )
)

optimized_complex_df = (
    orders_df
    .filter(F.col("amount") >= 800)
    .select("customer_id", "product_id", "amount")
    .join(
        F.broadcast(
            customers_df
            .filter(F.col("segment") == "Premium")
            .select("customer_id", "city")
        ),
        "customer_id"
    )
    .join(
        F.broadcast(
            products_df.select("product_id", "brand")
        ),
        "product_id"
    )
    .groupBy("city", "brand")
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("orders")
    )
)

print("COMPLEX BASELINE PLAN")
complex_baseline_df.explain("formatted")

print("\nOPTIMIZED COMPLEX PLAN")
optimized_complex_df.explain("formatted")

baseline_rows = sorted(complex_baseline_df.collect())
optimized_rows = sorted(optimized_complex_df.collect())

print("\nResults identical:", baseline_rows == optimized_rows)

# Example 49 — Production Troubleshooting Workflow

**Concepts:** evidence-driven optimization, explain plans, partition counts, skew, caching, configuration.

This example creates a reusable diagnostic workflow.

It does not claim to automatically diagnose every production problem. Instead, it demonstrates a structured checklist:

1. establish the query
2. inspect row counts
3. inspect partition counts
4. inspect key distributions
5. inspect the physical plan
6. identify likely shuffles and joins
7. change one thing at a time
8. validate correctness

In [ ]:
def diagnose_dataframe(df, name, key_column=None):
    print("=" * 80)
    print(f"DIAGNOSTIC: {name}")
    print("=" * 80)

    print("\nPartitions:", df.rdd.getNumPartitions())

    print("\nPhysical plan:")
    df.explain("formatted")

    if key_column is not None and key_column in df.columns:
        print(f"\nTop values for potential skew key: {key_column}")
        (
            df.groupBy(key_column)
            .count()
            .orderBy(F.desc("count"))
            .show(10, truncate=False)
        )

problem_query_df = (
    orders_df
    .join(products_df, "product_id")
    .filter(F.col("amount") > 300)
    .groupBy("customer_id", "category")
    .agg(F.sum("amount").alias("total_amount"))
)

diagnose_dataframe(problem_query_df, "problem_query_df", "customer_id")

print("\nTroubleshooting sequence:")
print("1. Check correctness and baseline metrics.")
print("2. Look for Exchange, Sort, Python UDF, and expensive joins.")
print("3. Check skew and partition imbalance.")
print("4. Check whether data layout can eliminate repeated work.")
print("5. Apply one targeted optimization.")
print("6. Re-check plan and Spark UI metrics.")
print("7. Keep the optimization only if it improves the real workload.")

# Example 50 — End-to-End Optimization Case Study

**Concepts combined:**

- early filtering
- projection pruning
- broadcast joins
- partition-aware data
- aggregation
- window functions
- AQE
- plan inspection
- correctness validation

This final example uses a small analytical scenario:

> Calculate premium-customer revenue for Electronics orders above a threshold, by region and customer, and rank customers within each region.

The workflow is:

1. baseline query
2. optimized query
3. compare plans
4. compare results

This is the pattern to carry into real production workloads.

In [ ]:
original_aqe = spark.conf.get("spark.sql.adaptive.enabled")

try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")

    # Baseline: joins first, filters and projections later.
    baseline_case_df = (
        orders_df
        .join(customers_df, "customer_id")
        .join(products_df, "product_id")
        .filter(
            (F.col("segment") == "Premium") &
            (F.col("category") == "Electronics") &
            (F.col("amount") >= 500)
        )
        .groupBy("region", "customer_id")
        .agg(F.sum("amount").alias("revenue"))
    )

    baseline_ranked_df = (
        baseline_case_df
        .withColumn(
            "regional_rank",
            F.dense_rank().over(
                Window.partitionBy("region").orderBy(F.desc("revenue"))
            )
        )
    )

    # Optimized: filter and project early, broadcast small filtered dimensions.
    optimized_orders_df = (
        orders_df
        .filter(F.col("amount") >= 500)
        .select("customer_id", "product_id", "amount", "region")
    )

    premium_customers_df = (
        customers_df
        .filter(F.col("segment") == "Premium")
        .select("customer_id")
    )

    electronics_products_df = (
        products_df
        .filter(F.col("category") == "Electronics")
        .select("product_id")
    )

    optimized_case_df = (
        optimized_orders_df
        .join(F.broadcast(premium_customers_df), "customer_id")
        .join(F.broadcast(electronics_products_df), "product_id")
        .groupBy("region", "customer_id")
        .agg(F.sum("amount").alias("revenue"))
    )

    optimized_ranked_df = (
        optimized_case_df
        .withColumn(
            "regional_rank",
            F.dense_rank().over(
                Window.partitionBy("region").orderBy(F.desc("revenue"))
            )
        )
    )

    print("BASELINE CASE STUDY PLAN")
    baseline_ranked_df.explain("formatted")

    print("\nOPTIMIZED CASE STUDY PLAN")
    optimized_ranked_df.explain("formatted")

    baseline_result = sorted(baseline_ranked_df.collect())
    optimized_result = sorted(optimized_ranked_df.collect())

    print("\nResults identical:", baseline_result == optimized_result)
    print("\nFinal optimized result:")
    optimized_ranked_df.orderBy("region", "regional_rank").show()

finally:
    spark.conf.set("spark.sql.adaptive.enabled", original_aqe)

# Series Summary — Examples 1–50

Congratulations — this completes the five-notebook Advanced PySpark Optimization series.

## The optimization progression

### 1–10 — Foundations and execution behavior

Core transformations, actions, lazy evaluation, narrow/wide transformations, partitions, and execution plans.

### 11–20 — Data layout and aggregation

Partitioning, pruning, bucketing, shuffle behavior, aggregation, multi-stage aggregation, rollup, and cube.

### 21–30 — Memory and joins

Caching, persistence, storage levels, memory awareness, spill observability, join planning, broadcast joins, and thresholds.

### 31–40 — Advanced joins and optimizer visibility

Sort-merge joins, bucket-aware joins, repartitioning, skew detection, salting, Catalyst, pushdown, pruning, and UDF trade-offs.

### 41–50 — Adaptive and production optimization

AQE, adaptive coalescing, adaptive skew handling, dynamic partition pruning, runtime filtering concepts, windows, complex plans, troubleshooting, and end-to-end optimization.

---

# Final Production Checklist

Before optimizing:

1. **Measure the real bottleneck.**
2. **Inspect the physical plan.**
3. **Look for Exchange, Sort, joins, Python UDFs, and repeated scans.**
4. **Check partition counts and key skew.**
5. **Understand data sizes before choosing a join strategy.**
6. **Prefer early filters and narrow projections.**
7. **Use built-in Spark functions when possible.**
8. **Cache only when reuse justifies it.**
9. **Use AQE where appropriate and inspect the final executed behavior.**
10. **Change one thing at a time.**
11. **Validate correctness after every optimization.**
12. **Keep optimizations that improve the production workload—not just a toy benchmark.**

The central lesson across all 50 examples is:

> **Spark optimization is primarily about understanding data movement, execution plans, and runtime behavior—not memorizing isolated APIs.**